# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset ([source](https://doi.org/10.71728/senscience.vq4a-28xa)) using the `mlcroissant` library. 

### Dataset Source
The dataset package is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json). All dataset elements are referenced by their `@id`.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

`mlcroissant` exposes the dataset structure in terms of record sets and fields. Each is accessible by their `@id`.

In [ ]:
# List available record sets and their fields
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"Record Set: {rs.name} (@id={rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id={field.id}, dataType={field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this example, we will extract from the main record set, which likely contains protein measurements. Substitute the correct `@id` as found above.

In [ ]:
# Identify RECORD SET @id to use (adapt to actual value from overview above)
# Use the first record set (as typical for FAIR2 packages)
record_set_ids = [rs.id for rs in metadata.record_sets]
print("Available record_set @id(s):", record_set_ids)
# We'll use the first for demonstration; replace if more suitable is found!
main_record_set_id = record_set_ids[0]

# Load all records (rows) from the main record set
df = pd.DataFrame(list(dataset.records(record_set=main_record_set_id)))
print(f"Columns in DataFrame (field @id):\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing numeric fields, or grouping by key categorical attributes.

In this section:
- We will select a representative numeric field for demonstration.
- Filter out lower-valued items.
- Normalize the numeric field.
- Group by a relevant categorical field if present.

In [ ]:
# Choose a numeric field and a group field by their @id (from earlier overview)
# If unsure, show up to 10 rows to inspect the column (field @id)s
print(df.head(10))
# Example: Let's assume 'cr:field_coverage_percent' and 'cr:field_accession' are available
# Replace with actual @id(s) observed in the DataFrame

numeric_field = None
group_field = None
# Try to find possible fields programmatically
for col in df.columns:
    # Example heuristic: if column name contains 'coverage' or a known numeric field
    if 'coverage' in col or 'intensity' in col or 'mw' in col.lower():
        numeric_field = col
        break
if not numeric_field and len(df.columns) > 0:
    # Just pick the first if nothing matches
    numeric_field = df.columns[0]

# Try to find a group field (e.g. protein or ID)
for col in df.columns:
    if 'accession' in col.lower() or 'protein' in col.lower() or 'description' in col.lower():
        group_field = col
        break

print(f"Using numeric field (for demonstration): '{numeric_field}'")
print(f"Using group field (for demo): '{group_field}'")

# Drop records with missing values in the numeric column
df_valid = df.dropna(subset=[numeric_field])

# Try to convert to float
df_valid[numeric_field] = pd.to_numeric(df_valid[numeric_field], errors='coerce')
threshold = df_valid[numeric_field].quantile(0.5)  # median value as threshold
filtered_df = df_valid[df_valid[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize (z-score) the numeric column
col_norm = numeric_field + '_normalized'
filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} (z-score):")
print(filtered_df[[numeric_field, col_norm]].head())

if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped (mean) {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and optionally its relationship with the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field is categorical with not too many categories, show boxplot
if group_field and filtered_df[group_field].nunique() <= 20:
    plt.figure(figsize=(10,6))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We have loaded and explored the FAIR² mass spectrometry protein dataset using `mlcroissant`, referencing all components by their Croissant schema `@id`.

- Inspected available record sets and field identifiers.
- Loaded records for the main record set.
- Performed exploratory analysis, filtering, and normalization on a numeric field.
- Visualized the distribution of protein attributes.

Further analysis might include investigating relationships among protein attributes, or integrating this dataset with additional biological knowledge bases. All data operations should be referenced using Croissant `@id`s to ensure schema compatibility and reproducibility.